# Voice_Cloning on Kaggle GPU

Kaggle notebooks don't get your local `.env`, so this notebook uses the bundled helpers (`kaggle/setup_kaggle.sh`, `kaggle/benchmark_kaggle.py`) which set every knob inside the process and benchmark **TTS generation + speed** on the GPU.

Kaggle has **no microphone or browser**, so live voice chat can't be tested here — only voice cloning + RTF. Use a local Chrome/Edge for the mic UI.

**Before running:** the code must be available as a Kaggle Dataset, or public on GitHub (set `REPO_URL` in the first cell).

In [ ]:
import glob
import os
import subprocess

# 1) Get the code onto the GPU machine --------------------------------
# Option A: attach this repo as a Kaggle Dataset (Add input) -> nothing to do.
# Option B: public GitHub repo -> set the URL and run this cell.
REPO_URL = ""  # e.g. "https://github.com/<you>/<repo>.git"

cands = glob.glob("/kaggle/working/Voice_Cloning") + glob.glob("/kaggle/input/*/Voice_Cloning")
if not cands and REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], cwd="/kaggle/working", check=True)
    cands = glob.glob("/kaggle/working/*/Voice_Cloning")
if not cands:
    raise SystemExit("Voice_Cloning folder not found - attach the repo as a Dataset or set REPO_URL.")
root = cands[0]
os.chdir(root)
print("repo root:", root)

## 2) Install dependencies + check CUDA
`requirements.txt` leaves Kaggle's preinstalled torch untouched.

In [ ]:
!python kaggle/setup_kaggle.sh

## 3) Benchmark `num_step` (16 / 24 / 32)
Each run downloads the OmniVoice weights on first execution, then prints inference time, audio length and **RTF**. Pick the highest `num_step` whose RTF stays below ~1 for your quality/speed sweet spot.

In [ ]:
!python kaggle/benchmark_kaggle.py

## 4) Listen to the outputs

In [ ]:
from IPython.display import Audio, display

import glob

for f in sorted(glob.glob("output_step*.wav")):
    print(f)
    display(Audio(f))

## Notes

- Reference voice: `my_voice.wav` is git-ignored (it is your recording). Upload a clean 3-10 s clip as `my_voice.wav` in the same folder before benchmarking, or set `VOICE_REF_AUDIO` in `benchmark_kaggle.py`.
- Chat (LLM replies) needs `MISTRAL_API_KEY`. Add it via **Add-ons ▸ Secrets** and expose it in a cell with:
  ```python
  from kaggle_secrets import UserSecretsClient
  os.environ["MISTRAL_API_KEY"] = UserSecretsClient().get_secret("MISTRAL_API_KEY")
  ```
- Edits in the notebook are NOT saved back to the repo: benchmark tuning here, keep the tuned `.env`/code locally.